# PIPELINE CREATION, RETRAINING AND EXECUTION SCRIPTS

## FOR FASTAPI (AI CORPORATE SUITE)

In [1]:
import os
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

PROJECT_PATH = '/Users/rober/smartport-ai-risk-early-warning'
MODELS_PATH = os.path.join(PROJECT_PATH, '04_Models')

# Load balanced data from CSV
X_BALANCED = os.path.join(PROJECT_PATH, '02_Data/03_Working/X_balanced.csv')
Y_BALANCED = os.path.join(PROJECT_PATH, '02_Data/03_Working/y_balanced.csv')

print("Loading balanced data from CSV...")
X_res = pd.read_csv(X_BALANCED)
y_res = pd.read_csv(Y_BALANCED).squeeze()

print(f"Data shape: X={X_res.shape}, y={y_res.shape}")

# Create FRESH pipeline
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# Train
print("Training model...")
pipe.fit(X_res, y_res)

# Save
EXECUTION_PIPE = os.path.join(MODELS_PATH, 'pipe_execution.pkl')
FEATURES_FILE = os.path.join(MODELS_PATH, 'model_features.pkl')

joblib.dump(pipe, EXECUTION_PIPE)
joblib.dump(X_res.columns.tolist(), FEATURES_FILE)

print(f"✔ Model saved: {EXECUTION_PIPE}")
print(f"✔ Features saved: {FEATURES_FILE}")

/var/folders/6c/byy38myn1t50449jbzfjs1_c0000gn/T/ipykernel_22407/3887420565.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Loading balanced data from CSV...
Data shape: X=(138896, 11), y=(138896,)
Training model...
✔ Model saved: /Users/rober/smartport-ai-risk-early-warning/04_Models/pipe_execution.pkl
✔ Features saved: /Users/rober/smartport-ai-risk-early-warning/04_Models/model_features.pkl


## PIPELINE CREATION

In [5]:
import os
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# Project Paths
PROJECT_PATH = '/Users/rober/smartport-ai-risk-early-warning'
MODELS_PATH = os.path.join(PROJECT_PATH, '04_Models')

if not os.path.exists(MODELS_PATH):
    os.makedirs(MODELS_PATH)

# Create FRESH pipeline with UNTRAINED XGBoost
pipe_retraining = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# Save the UNTRAINED pipeline skeleton
pipe_retraining_path = os.path.join(MODELS_PATH, 'pipe_retraining.pkl')
joblib.dump(pipe_retraining, pipe_retraining_path)

print(f"✔ Success: pipe_retraining.pkl (UNTRAINED skeleton) created at {pipe_retraining_path}")

✔ Success: pipe_retraining.pkl (UNTRAINED skeleton) created at /Users/rober/smartport-ai-risk-early-warning/04_Models/pipe_retraining.pkl


## RETRAINING SCRIPT

In [6]:
# ==============================================================================
# AUTOMATED RETRAINING SCRIPT
# ==============================================================================
import os
import pandas as pd
import joblib
from sklearn.metrics import recall_score

def run_retraining():
    BASE_PATH = '/Users/rober/smartport-ai-risk-early-warning'
    
    # Use the BALANCED data from the balancing notebook
    X_BALANCED = os.path.join(BASE_PATH, '02_Data/03_Working/X_balanced.pickle')
    Y_BALANCED = os.path.join(BASE_PATH, '02_Data/03_Working/y_balanced.pickle')
    
    SKELETON = os.path.join(BASE_PATH, '04_Models/pipe_retraining.pkl')
    EXECUTION_PIPE = os.path.join(BASE_PATH, '04_Models/pipe_execution.pkl')
    FEATURES_FILE = os.path.join(BASE_PATH, '04_Models/model_features.pkl')

    print("Starting retraining process...")
    
    # 1. Load BALANCED Data (already SMOTE-Tomek processed)
    print("Loading balanced data from previous notebook...")
    X_res = pd.read_pickle(X_BALANCED)
    y_res = pd.read_pickle(Y_BALANCED)
    
    print(f"   Data shape: X={X_res.shape}, y={y_res.shape}")
    print(f"   Class distribution: {y_res.value_counts().to_dict()}")
    
    # 2. Load Pipeline Skeleton
    print("Loading pipeline skeleton...")
    pipe = joblib.load(SKELETON)
    
    # 3. Train (re-fit the pipeline on new balanced data)
    print("Training pipeline on balanced data...")
    pipe.fit(X_res, y_res)
    
    # 4. Audit: Ensure the model catches at least 90% of delays
    probs = pipe.predict_proba(X_res)[:, 1]
    recall = recall_score(y_res, (probs >= 0.5).astype(int))
    
    if recall >= 0.90:
        # Save TRAINED execution pipeline
        joblib.dump(pipe, EXECUTION_PIPE)
        
        # Save feature list for column alignment in production
        joblib.dump(X_res.columns.tolist(), FEATURES_FILE)
        
        print(f"✔ Retraining successful. Model promoted with Recall: {recall:.4f}")
        print(f"✔ Execution pipeline saved: {EXECUTION_PIPE}")
        print(f"✔ Feature list saved: {FEATURES_FILE}")
    else:
        print(f"✘ Retraining failed. Recall {recall:.4f} is too low (minimum: 0.90)")

if __name__ == "__main__":
    run_retraining()

Starting retraining process...
Loading balanced data from previous notebook...
   Data shape: X=(138896, 11), y=(138896,)
   Class distribution: {0.0: 69448, 1.0: 69448}
Loading pipeline skeleton...
Training pipeline on balanced data...


/Users/rober/opt/miniconda3/envs/smartport/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [16:42:05] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✔ Retraining successful. Model promoted with Recall: 0.9646
✔ Execution pipeline saved: /Users/rober/smartport-ai-risk-early-warning/04_Models/pipe_execution.pkl
✔ Feature list saved: /Users/rober/smartport-ai-risk-early-warning/04_Models/model_features.pkl


## EXECUTION SCRIPT

In [7]:
# ==============================================================================
# EXECUTION SCRIPT (PRODUCTION)
# ==============================================================================
import os
import pandas as pd
import joblib
from datetime import datetime

# Business Decisions based on Risk Levels
ACTION_MAP = {
    'CRITICAL': 'IMMEDIATE: Priority berthing & Tugboat standby.',
    'WARNING': 'PROACTIVE: Verify ETA and terminal capacity.',
    'NORMAL': 'ROUTINE: Follow standard operations.'
}

def run_execution():
    BASE_PATH = '/Users/rober/smartport-ai-risk-early-warning'
    MODEL_FILE = os.path.join(BASE_PATH, '04_Models/pipe_execution.pkl')
    FEATURES_FILE = os.path.join(BASE_PATH, '04_Models/model_features.pkl')
    SOURCE_DATA = os.path.join(BASE_PATH, '02_Data/03_Working/work_fs.csv')  # Simulating new data
    OUTPUT_CSV = os.path.join(BASE_PATH, '05_Outputs/risk_alerts.csv')

    if not os.path.exists(MODEL_FILE):
        print("✘ Error: Execution model not found. Run retraining first.")
        return

    # 1. Load TRAINED Pipeline and Expected Features
    pipe = joblib.load(MODEL_FILE)
    expected_features = joblib.load(FEATURES_FILE)
    
    print(f"✔ Model loaded: {MODEL_FILE}")
    print(f"✔ Expected features ({len(expected_features)}): {expected_features[:5]}...")
    
    # 2. Load New Data
    df = pd.read_csv(SOURCE_DATA)
    X_live = df.drop(columns=['delay_flag']) if 'delay_flag' in df.columns else df
    
    # 3. Align columns with training (handle missing/extra columns)
    for col in expected_features:
        if col not in X_live.columns:
            X_live[col] = 0
    
    X_live = X_live[expected_features]
    
    # 4. Predict (pipeline handles imputation automatically)
    probs = pipe.predict_proba(X_live)[:, 1]
    
    # 5. Alert Logic
    results = pd.DataFrame({
        'vessel_index': df.index,
        'risk_score': probs
    })
    
    # Thresholds calibrated for balanced XGBoost
    results['risk_level'] = results['risk_score'].apply(
        lambda x: 'CRITICAL' if x >= 0.90 else (
                   'WARNING' if x >= 0.70 else 'NORMAL')
    )
    results['recommended_action'] = results['risk_level'].map(ACTION_MAP)
    results['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # 6. Export
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    results.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\n✔ Execution completed. Alerts saved: {OUTPUT_CSV}")
    print(f"   Distribution: {results['risk_level'].value_counts().to_dict()}")

if __name__ == "__main__":
    run_execution()

✔ Model loaded: /Users/rober/smartport-ai-risk-early-warning/04_Models/pipe_execution.pkl
✔ Expected features (11): ['day_of_week', 'rolling_mean_sog', 'hdg', 'movement_stability', 'cog']...

✔ Execution completed. Alerts saved: /Users/rober/smartport-ai-risk-early-warning/05_Outputs/risk_alerts.csv
   Distribution: {'CRITICAL': 65561, 'NORMAL': 33006, 'WARNING': 17914}
